<a href="https://vigneashpandiyan.github.io/publications/Codes/" target="_blank" rel="noopener noreferrer">
  <img src="https://vigneashpandiyan.github.io/images/Link.png"
       style="max-width: 800px; width: 100%; height: auto;">
</a>

# Segmentation: Pixel-level Classification

Image segmentation is a computer vision task where you assign a label to each pixel in an image, instead of predicting a single label for the whole image.

In [ ]:
import numpy as np
from torch.utils.data import DataLoader
import torch.optim as optim
import torch.nn as nn
import matplotlib.pyplot as plt
import torch
import torchvision
from torchvision import transforms
from torchvision.datasets import OxfordIIITPet

We will use the Oxford-IIIT Pet Dataset which is a popular benchmark for pet classification and segmentation.

It contains about 7000+ images of 37 pet breeds (cats and dogs), with variations in pose, lighting, and background. For segmentation, it provides pixel-level annotations: commonly used as foreground (pet) vs background masks, and also includes a “trimap” style boundary region.

In [ ]:
# Transformations: Resize to a standard power of 2 for the model
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])

# This will download the dataset directly into the Colab temporary storage
train_dataset = OxfordIIITPet(
    root='./data',
    split='trainval',
    target_types='segmentation',
    download=True,
    transform=transform,
    target_transform=transform
)

print(f"Dataset downloaded! Number of images: {len(train_dataset)}")

Preview of one image from dataset:

In [ ]:
# Get one sample
image, mask = train_dataset[0]

# Plot
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.title("Input Image")
plt.imshow(image.permute(1, 2, 0))

plt.subplot(1, 2, 2)
plt.title("Segmentation Mask")
plt.imshow(mask.squeeze(), cmap='gray')
plt.show()

## Setup Dataloaders

In [ ]:
# 1. SETUP DATASET AND LOADERS
# We resize to 128x128 to ensure it trains quickly during this demo
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])

# Custom target transform to convert Pet Dataset labels {1, 2, 3} to binary {0, 1}


def mask_to_binary(mask):
    mask = transforms.Resize((64, 64))(mask)
    mask = torch.as_tensor(np.array(mask), dtype=torch.int64)
    # The dataset uses: 1:Foreground, 2:Background, 3:Outline.
    # We'll treat Foreground (1) as our target class.
    binary_mask = (mask == 1).float()
    return binary_mask.unsqueeze(0)


print("Downloading dataset... (this may take a minute)")
train_data = OxfordIIITPet(root='./data', split='trainval', target_types='segmentation',
                           download=True, transform=transform, target_transform=mask_to_binary)
test_data = OxfordIIITPet(root='./data', split='test', target_types='segmentation',
                          download=True, transform=transform, target_transform=mask_to_binary)

train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
test_loader = DataLoader(test_data, batch_size=1, shuffle=False)
print(f"Dataset ready! Training samples: {len(train_data)}")

## Define Architecture

A simple U-Net is an encoder–decoder segmentation network with skip connections that copy high-resolution features from the encoder to the decoder, helping recover fine boundaries.

In [ ]:
# 2. DEFINE THE UNET ARCHITECTURE


class SimpleUNet(nn.Module):
    def __init__(self):
        super(SimpleUNet, self).__init__()
        # Encoder (Downsampling)
        self.enc1 = nn.Conv2d(3, 16, kernel_size=3,
                              padding=1)  # 3 channels for RGB
        self.enc2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)

        # Bottleneck
        self.bottleneck = nn.Conv2d(32, 64, kernel_size=3, padding=1)

        # Decoder (Upsampling)
        self.up2 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec2 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.up1 = nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2)
        self.dec1 = nn.Conv2d(16, 16, kernel_size=3, padding=1)

        self.final = nn.Conv2d(16, 1, kernel_size=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        e1 = torch.relu(self.enc1(x))
        e2 = torch.relu(self.enc2(self.pool(e1)))
        b = torch.relu(self.bottleneck(self.pool(e2)))
        d2 = torch.relu(self.dec2(self.up2(b)))
        d1 = torch.relu(self.dec1(self.up1(d2)))
        return self.sigmoid(self.final(d1))


# Initialize Model, Loss, and Optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleUNet().to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

## Training

In [ ]:

# 3. TRAINING LOOP
print("\nStarting Training...")
model.train()
for epoch in range(100):  # Increase epochs for better results
    running_loss = 0.0
    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/100], Loss: {running_loss/len(train_loader):.4f}")

## Visualization

The Dice coefficient (a.k.a. Sørensen–Dice) measures how well a predicted segmentation mask overlaps the ground-truth mask. It ranges from 0 (no overlap) to 1 (perfect match).

In [ ]:
# 4. METRICS & VISUALIZATION


def dice_coeff(pred, target, smooth=1.0):
    pred = (pred > 0.5).float()
    intersection = (pred * target).sum()
    return (2. * intersection + smooth) / (pred.sum() + target.sum() + smooth)


print("\nEvaluating and Visualizing...")
model.eval()
with torch.no_grad():
    # Grab one sample from the test set
    img, mask = next(iter(test_loader))
    pred = model(img.to(device))
    score = dice_coeff(pred.cpu(), mask)

    # Convert to numpy for plotting
    img_np = img[0].permute(1, 2, 0).numpy()
    mask_np = mask[0].squeeze().numpy()
    pred_np = pred[0].cpu().squeeze().numpy()

    # Final Plot
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    ax[0].imshow(img_np)
    ax[0].set_title("Input Image")
    ax[1].imshow(mask_np, cmap='gray')
    ax[1].set_title("Ground Truth Mask")
    ax[2].imshow(pred_np, cmap='gray')
    ax[2].set_title(f"Prediction (Dice: {score:.2f})")
    plt.tight_layout()
    plt.show()